# ControlNet training test

Этот ноутбук проверяет, что пайплайн обучения `controlnet` запускается end-to-end:
1. Подбирает путь к pretrained EMA UNet.
2. Собирает датасеты и dataloader'ы.
3. Создает `ControlledUNet` и `ControlNetTrainer`.
4. Запускает `trainer.train_loop()`.

По умолчанию выставлены щадящие параметры для smoke-test. Для полноценного обучения увеличьте `num_epochs` и `train_batch_size`.

In [1]:
import os
import glob
import json
import platform
import logging

import torch

if platform.system() == 'Darwin':
    REPO_DIR = '/Users/amir/sciml/diffusion_data_assimilation'
    DATA_DIR = '/Users/amir/sciml/sea_ice_data'
else:
    REPO_DIR = '/home'
    DATA_DIR = '/mnt/sciml/a.sadreev/sea_ice_data'

os.chdir(REPO_DIR)

PRETRAINED_UNET_PATH = "/home/checkpoints/best_model/ema_state.pth"

if PRETRAINED_UNET_PATH is None:
    candidates = sorted(glob.glob(os.path.join(REPO_DIR, 'checkpoints', '**', 'ema_best_model.pth'), recursive=True))
    assert candidates, 'Не найден ни один ema_best_model.pth в checkpoints/**'
    PRETRAINED_UNET_PATH = candidates[-1]

print(f'cwd: {os.getcwd()}')
print(f'data: {DATA_DIR}')
print(f'pretrained unet: {PRETRAINED_UNET_PATH}')
print(f'cuda available: {torch.cuda.is_available()}')

cwd: /home
data: /mnt/sciml/a.sadreev/sea_ice_data
pretrained unet: /home/checkpoints/best_model/ema_state.pth
cuda available: True


In [2]:
import numpy as np
from functools import partial

from diffusers.models.unets.unet_2d import UNet2DModel
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_cosine_schedule_with_warmup

from utils import NpyImageDataset, MixedSatelliteTrackDataset, channel_normalize, add_noise
from controlnet.model import SeaIceControlNet, ControlledUNet
from controlnet.trainer import ControlNetTrainingConfig, ControlNetTrainer

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')
assert torch.cuda.is_available(), 'Для этого ноутбука требуется CUDA (как в controlnet/main.py)'

In [3]:
config = ControlNetTrainingConfig(
    data_dir_train=f'{DATA_DIR}/train',
    data_dir_valid=f'{DATA_DIR}/valid',
    data_dir_satellite_mask=f'{DATA_DIR}/satellite_samples',
    pretrained_unet_path=PRETRAINED_UNET_PATH,
    train_batch_size=24,
    eval_batch_size=100,
    num_epochs=10,
    push_to_hub=False,
)

print(json.dumps({
    'data_dir_train': config.data_dir_train,
    'data_dir_valid': config.data_dir_valid,
    'data_dir_satellite_mask': config.data_dir_satellite_mask,
    'pretrained_unet_path': config.pretrained_unet_path,
    'train_batch_size': config.train_batch_size,
    'eval_batch_size': config.eval_batch_size,
    'num_epochs': config.num_epochs,
    'push_to_hub': config.push_to_hub,
}, indent=2))

{
  "data_dir_train": "/mnt/sciml/a.sadreev/sea_ice_data/train",
  "data_dir_valid": "/mnt/sciml/a.sadreev/sea_ice_data/valid",
  "data_dir_satellite_mask": "/mnt/sciml/a.sadreev/sea_ice_data/satellite_samples",
  "pretrained_unet_path": "/home/checkpoints/best_model/ema_state.pth",
  "train_batch_size": 24,
  "eval_batch_size": 100,
  "num_epochs": 10,
  "push_to_hub": false
}


In [4]:
transform = partial(
    channel_normalize,
    channel_mean=config.channel_mean,
    channel_std=config.channel_std,
)

use_pin_memory = torch.cuda.is_available()

dataset_train = NpyImageDataset(
    folder=config.data_dir_train,
    transform=transform,
    preload=False,
    mmap_mode='r',
)
train_dataloader = torch.utils.data.DataLoader(
    dataset_train,
    batch_size=config.train_batch_size,
    shuffle=True,
    num_workers=config.num_workers_train,
    pin_memory=use_pin_memory,
    persistent_workers=True,
    prefetch_factor=2,
    drop_last=True,
)

dataset_valid = NpyImageDataset(
    folder=config.data_dir_valid,
    transform=transform,
    preload=False,
    mmap_mode='r',
)
valid_dataloader = torch.utils.data.DataLoader(
    dataset_valid,
    batch_size=config.eval_batch_size,
    shuffle=False,
    num_workers=config.num_workers_val,
    pin_memory=use_pin_memory,
    persistent_workers=True,
    prefetch_factor=2,
)

valid_mask_path = os.path.join(os.path.dirname(config.data_dir_train), 'mask_padding.npy')
valid_mask = np.load(valid_mask_path).astype(np.float32)

dataset_satellite_mask = MixedSatelliteTrackDataset(
    folder=config.data_dir_satellite_mask,
    image_size=config.image_size,
    valid_mask=valid_mask,
    npy_fraction=0.5,
    generate_fraction=0.3,
    empty_fraction=0.2,
    n_tracks_range=config.satellite_n_tracks_range,
    mmap_mode='r',
)
satellite_mask_dataloader = torch.utils.data.DataLoader(
    dataset_satellite_mask,
    batch_size=config.train_batch_size,
    shuffle=False,
    num_workers=config.num_workers_val,
    pin_memory=use_pin_memory,
    persistent_workers=True,
    prefetch_factor=2,
)

print(f'train samples: {len(dataset_train)} | valid samples: {len(dataset_valid)}')
print(f'train batches: {len(train_dataloader)} | valid batches: {len(valid_dataloader)}')

train samples: 60816 | valid samples: 8760
train batches: 2534 | valid batches: 88


In [5]:
# Базовый UNet (та же архитектура, что в gradient_based)
unet = UNet2DModel(
    sample_size=config.image_size,
    in_channels=config.in_channels,
    out_channels=config.out_channels,
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 512, 512),
    down_block_types=(
        'DownBlock2D', 'DownBlock2D', 'DownBlock2D',
        'AttnDownBlock2D', 'DownBlock2D',
    ),
    up_block_types=(
        'UpBlock2D', 'AttnUpBlock2D', 'UpBlock2D',
        'UpBlock2D', 'UpBlock2D',
    ),
)

# Загружаем EMA-веса базовой diffusion модели
ema = EMAModel(unet.parameters(), decay=0.999)
try:
    ema_state = torch.load(config.pretrained_unet_path, map_location='cpu', weights_only=True)
except TypeError:
    ema_state = torch.load(config.pretrained_unet_path, map_location='cpu')
ema.load_state_dict(ema_state)
ema.copy_to(unet.parameters())

controlnet = SeaIceControlNet(
    unet,
    conditioning_channels=config.controlnet_conditioning_channels,
)
model = ControlledUNet(unet, controlnet)

optimizer = torch.optim.AdamW(controlnet.parameters(), lr=config.learning_rate)
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=config.lr_warmup_steps,
    num_training_steps=len(train_dataloader) * config.num_epochs,
)

trainer = ControlNetTrainer(
    config=config,
    model=model,
    optimizer=optimizer,
    data_loader_train=train_dataloader,
    data_loader_val=valid_dataloader,
    data_loader_satellite_mask=satellite_mask_dataloader,
    lr_scheduler=lr_scheduler,
    add_noise_func=add_noise,
)

print('Trainer собран успешно')

Trainer собран успешно


In [ ]:
# Запуск обучения
trainer.train_loop()

  0%|          | 0/2534 [00:00<?, ?it/s]

2026-04-03 08:08:34,403 [INFO] controlnet.trainer: Epoch 0 — val_loss: 0.065236  full: 0.058864  masked: 0.063718
2026-04-03 08:08:38,468 [INFO] controlnet.trainer: Sample saved: checkpoints/controlnet/run_20260403_071713/samples/epoch_0000.png
2026-04-03 08:08:39,282 [INFO] controlnet.trainer: New best model saved! val_loss: 0.065236


  0%|          | 0/2534 [00:00<?, ?it/s]